In [40]:
import numpy as np # Import NumPy for numerical operations, aliased as 'np'
import pandas as pd # Import Pandas for data manipulation and analysis, aliased as 'pd'
import torch # Import PyTorch, an open-source machine learning framework

In [41]:
import kagglehub # Import kagglehub for easy access to Kaggle datasets

# Download the latest version of the specified dataset
path = kagglehub.dataset_download("quadeer15sh/celeba-face-recognition-triplets")

# Print the local path where the dataset files are downloaded
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'celeba-face-recognition-triplets' dataset.
Path to dataset files: /kaggle/input/celeba-face-recognition-triplets


In [42]:
import os # Import the os module for interacting with the operating system
print(os.listdir(path)) # List the contents of the downloaded dataset directory

['CelebAAttrs.csv', 'CelebA FR Triplets']


In [43]:
path = kagglehub.dataset_download("quadeer15sh/celeba-face-recognition-triplets")

# Construct the correct path to the 'CelebA FR Triplets' subdirectory
correct_path = os.path.join(path, "CelebA FR Triplets")
# List the contents of the 'CelebA FR Triplets' subdirectory
print(os.listdir(correct_path))

Using Colab cache for faster access to the 'celeba-face-recognition-triplets' dataset.
['CelebA FR Triplets']


In [44]:
import os, kagglehub

# Download the dataset again (if not already cached)
path = kagglehub.dataset_download("quadeer15sh/celeba-face-recognition-triplets")

# Construct the root directory path to the actual data, navigating through nested folders
root_dir = os.path.join(path, "CelebA FR Triplets", "CelebA FR Triplets")

# Print the determined root directory and list its contents (images and triplets.csv)
print("Root dir:", root_dir)
print("Files:", os.listdir(root_dir))

Using Colab cache for faster access to the 'celeba-face-recognition-triplets' dataset.
Root dir: /kaggle/input/celeba-face-recognition-triplets/CelebA FR Triplets/CelebA FR Triplets
Files: ['triplets.csv', 'images']


In [45]:
from torch.utils.data import Dataset, DataLoader

# Define a custom Dataset class for handling triplet images
class TripletDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir  # Root directory containing the dataset
        self.image_dir = os.path.join(root_dir, "images") # Directory containing image files
        # Load the triplets annotations from 'triplets.csv' into a pandas DataFrame
        self.annotations = pd.read_csv(os.path.join(root_dir, "triplets.csv"))
        self.transform = transform  # Transformations to be applied to images

    def __len__(self):
        # Return the total number of triplets in the dataset
        return len(self.annotations)

    def __getitem__(self, index):
        # Get the filenames for anchor, positive, and negative images for a given index
        anchor_img = self.annotations.iloc[index, 0]
        positive_img = self.annotations.iloc[index, 2]
        negative_img = self.annotations.iloc[index, 4]

        # Open the images and convert them to RGB format
        anc = Image.open(os.path.join(self.image_dir, anchor_img)).convert("RGB")
        pos = Image.open(os.path.join(self.image_dir, positive_img)).convert("RGB")
        neg = Image.open(os.path.join(self.image_dir, negative_img)).convert("RGB")

        # Apply transformations if provided
        if self.transform:
            anc = self.transform(anc)
            pos = self.transform(pos)
            neg = self.transform(neg)

        # Return the transformed anchor, positive, and negative images
        return anc, pos, neg

In [47]:
from sklearn.model_selection import train_test_split
import torchvision.transforms as transforms
from torch.utils.data import Subset
from PIL import Image

# Define a series of transformations to apply to the images
transform = transforms.Compose([
    transforms.Resize((112, 112)),  # Resize images to 112x112 pixels
    transforms.ToTensor(),          # Convert images to PyTorch tensors
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3) # Normalize pixel values to range [-1, 1]
])

# Create an instance of the TripletDataset with the specified root directory and transformations
dataset = TripletDataset(root_dir=root_dir, transform=transform)

# Split the dataset into training and testing subsets
train_indice, test_indice = train_test_split(list(range(len(dataset))),test_size=0.2, random_state=42)
# Create Subset objects for training and testing data
train_subset, test_subset = Subset(dataset, train_indice), Subset(dataset, test_indice)

# Create DataLoader for the training set
train_loader = DataLoader(train_subset, batch_size=32, shuffle=True,  num_workers=2,pin_memory=True, prefetch_factor=2) # Reduced num_workers to 2 to avoid UserWarning
# Create DataLoader for the testing set
test_loader = DataLoader(test_subset, batch_size=32, shuffle=True)

# Print the shape of the second image (positive image) from the first triplet in the dataset
print(dataset[0][1].shape)

torch.Size([3, 112, 112])


In [48]:
import torch.nn as nn # Import nn module for neural network layers
import torch.nn.functional as F # Import functional module for activation functions, etc.
import torch # Import the main PyTorch library

In [49]:
class FaceRecognition(nn.Module):
    def __init__(self):
        super(FaceRecognition,self).__init__()

        # Convolutional layers with BatchNorm and MaxPool
        self.conv1 = nn.Conv2d(in_channels=3,out_channels=8, kernel_size=3, padding=1) #112*112 input image
        self.bn1 = nn.BatchNorm2d(8)

        self.conv2 = nn.Conv2d(in_channels=8,out_channels=16, kernel_size=3, padding=1) #56*56 after first pooling
        self.bn2 = nn.BatchNorm2d(16)

        self.conv3 = nn.Conv2d(in_channels=16,out_channels=32, kernel_size=3, padding=1) #28*28 after second pooling
        self.bn3 = nn.BatchNorm2d(32)

        self.conv4 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1) #14*14 after third pooling
        self.bn4 = nn.BatchNorm2d(64)

        self.conv5 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1) #7*7 after fourth pooling
        self.bn5 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2,2) # Max pooling layer with 2x2 kernel and stride 2

        # Fully connected layer. Input features derived from 128 channels * 3x3 feature map after 5 pooling layers
        self.fc1 = nn.Linear(in_features=128 * 3 * 3, out_features=512)

    def forward(self, x):
        # Apply Conv -> BatchNorm -> ReLU -> MaxPool for each block
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.pool(F.relu(self.bn5(self.conv5(x))))

        # Flatten the output for the fully connected layer
        x = torch.flatten(x, 1)
        # Apply the fully connected layer
        x = self.fc1(x)
        # Normalize the embedding to unit length (L2 norm)
        x = F.normalize(x, p=2, dim=1)

        return x

# Determine the device to use for training (GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [50]:
import torch.optim as optim
from numpy.linalg import norm as norm
import torch.nn as nn

# Instantiate the FaceRecognition model and move it to the appropriate device (CPU or GPU)
model = FaceRecognition().to(device)

NUM_EPOCHS = 25 # Define the total number of training epochs
criterion = nn.TripletMarginLoss(margin=0.2, p=2) # Initialize TripletMarginLoss with a margin of 0.2
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4) # Initialize Adam optimizer with learning rate and weight decay
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_loader) * NUM_EPOCHS) # Initialize CosineAnnealingLR scheduler

In [ ]:
NUM_EPOCHS = 25
model.train() # Set the model to training mode
for epochs in range(NUM_EPOCHS):
    running_loss = 0.0
    for i, data in enumerate(train_loader):
        # Move anchor, positive, and negative images to the device (CPU/GPU)
        anc_images, pos_images, neg_images = data[0].to(device), data[1].to(device), data[2].to(device)

        # Get embeddings for anchor, positive, and negative images
        anc_embedding = model(anc_images)
        pos_embedding = model(pos_images)
        neg_embedding = model(neg_images)

        # Zero the gradients before backpropagation
        optimizer.zero_grad()

        # Calculate the triplet loss
        loss = criterion(anc_embedding, pos_embedding, neg_embedding)

        # Perform backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        # Update learning rate scheduler
        scheduler.step()

        running_loss += loss.item()
        # Print loss every 100 steps
        if (i + 1) % 100 == 0:
            print(f'Epoch [{epochs + 1}/{NUM_EPOCHS}], Step [{i + 1}/{len(train_loader)}], Loss: {running_loss / 100:.4f}')
            running_loss = 0.0

print('Finished Training')

### Saved Model

In [ ]:
import torch

PATH = "facenet_model.pth"
torch.save(model.state_dict(), PATH)

print("Model saved successfully")

### Load model

In [11]:
from google.colab import files
uploaded = files.upload()

Saving facenet_model.pth to facenet_model.pth


In [12]:
model = FaceRecognition()   # create model object

PATH = "facenet_model.pth"
model.load_state_dict(torch.load(PATH, map_location=torch.device('cpu')))

<All keys matched successfully>

### Move Model to GPU

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

In [14]:
import torch

print("\n--- Running Simple Evaluation on Test Set ---")

model.eval()

total_loss = 0
correct_triplets = 0
total_triplets = 0


--- Running Simple Evaluation on Test Set ---


In [15]:
# Disable gradient computation for evaluation (saves memory and speeds up testing)
with torch.no_grad():

    # Loop through batches of triplets from test loader
    for data in test_loader:

        # Extract anchor, positive, and negative images and move them to device (GPU/CPU)
        anc_images = data[0].to(device)
        pos_images = data[1].to(device)
        neg_images = data[2].to(device)

        # Forward pass: generate embeddings for each image in the triplet
        anc_embedding = model(anc_images)
        pos_embedding = model(pos_images)
        neg_embedding = model(neg_images)

        # Compute Triplet Loss for the current batch
        loss = criterion(anc_embedding, pos_embedding, neg_embedding)

        # Accumulate total test loss
        total_loss += loss.item()

        # Compute Euclidean distance between anchor and positive embeddings
        dist_pos = torch.norm(anc_embedding - pos_embedding, dim=1)

        # Compute Euclidean distance between anchor and negative embeddings
        dist_neg = torch.norm(anc_embedding - neg_embedding, dim=1)

        # Count triplets where positive is closer to anchor than negative (correct prediction)
        correct_triplets += torch.sum(dist_pos < dist_neg).item()

        # Count total number of evaluated triplets
        total_triplets += anc_images.size(0)


# Compute average loss over all test batches
avg_test_loss = total_loss / len(test_loader)

# Compute triplet accuracy as percentage
accuracy = 100 * correct_triplets / total_triplets

# Print final average test loss
print(f"Average Test Loss: {avg_test_loss:.4f}")

Average Test Loss: 0.0715


In [16]:
print(f"Triplet Accuracy: {accuracy:.2f}%")

Triplet Accuracy: 85.06%


# VERIFICATION (1:1) — Same or Different Person

In [ ]:
def get_embedding(model, image, device):
    # Set model to evaluation mode (disables dropout and batchnorm training behavior)
    model.eval()

    # Disable gradient calculation for inference (saves memory and computation)
    with torch.no_grad():

        # Move input image tensor to the specified device (GPU or CPU)
        image = image.to(device)

        # Forward pass: generate embedding vector from the model
        embedding = model(image)

    # Return the computed embedding
    return embedding

In [ ]:
def verify(model, img1, img2, threshold, device):

    # Extract embedding for first image
    emb1 = get_embedding(model, img1, device)

    # Extract embedding for second image
    emb2 = get_embedding(model, img2, device)

    # Compute Euclidean (L2) distance between the two embeddings
    # Smaller distance means more similar faces
    distance = torch.norm(emb1 - emb2, p=2).item()

    # Compare distance with threshold to decide if same person
    if distance < threshold:
        return "Same Person", distance  # Faces are similar
    else:
        return "Different Person", distance  # Faces are different

In [ ]:
import cv2
img1 = cv2.imread("/content/one_is_one.jpg",cv2.COLOR_BGR2RGB)
img2 = cv2.imread("/content/one_is_one.jpg",cv2.COLOR_BGR2RGB)

In [ ]:
print(img1 is None)
print(img2 is None)

False
False


In [ ]:
img1 = cv2.resize(img1, (112, 112))
img2 = cv2.resize(img2, (112, 112))

In [ ]:
import torch
import numpy as np

def preprocess(img):
    img = img.astype(np.float32) / 255.0   # scale 0-1
    img = torch.tensor(img).permute(2, 0, 1)  # HWC → CHW
    img = img.unsqueeze(0)  # add batch dimension
    return img

In [ ]:
# Preprocess first image (resize, normalize, convert to tensor, add batch dimension)
img1 = preprocess(img1)

# Preprocess second image (resize, normalize, convert to tensor, add batch dimension)
img2 = preprocess(img2)

In [ ]:
# Move first image tensor to the selected device (GPU or CPU)
img1 = img1.to(device)

# Move second image tensor to the selected device (GPU or CPU)
img2 = img2.to(device)

In [ ]:
threshold = 0.8

result, distance = verify(model, img1, img2, threshold, device)

print("Result:", result)
print("Distance:", distance)

Result: Same Person
Distance: 0.0


### Identification: 1:N accuracy

In [17]:
import os

def build_gallery_simple(model, gallery_path, device):

    # Set model to evaluation mode (important for BatchNorm / Dropout)
    model.eval()

    # Dictionary to store identity name → embedding
    gallery = {}

    # Disable gradient computation for inference
    with torch.no_grad():

        # Loop through all files inside the gallery folder
        for img_name in os.listdir(gallery_path):

            # Create full image path
            img_path = os.path.join(gallery_path, img_name)

            # Read image using OpenCV
            img = cv2.imread(img_path)

            # Skip file if image could not be loaded
            if img is None:
                continue

            # Preprocess image (resize, normalize, convert to tensor, add batch dim)
            # Then move to device (GPU/CPU)
            img = preprocess(img).to(device)

            # Generate embedding using the model
            emb = model(img)

            # Remove file extension (.jpg, .png) to use filename as identity
            person_name = os.path.splitext(img_name)[0]

            # Store embedding in gallery dictionary
            gallery[person_name] = emb

    # Return dictionary containing all gallery embeddings
    return gallery

In [22]:
import os
import cv2
import torch
import numpy as np

# Preprocessing function to prepare image for the model
def preprocess(img):
    img = img.astype(np.float32) / 255.0   # Convert image to float32 and scale pixel values to [0, 1]
    img = torch.tensor(img).permute(2, 0, 1)  # Convert image from HWC format to CHW format
    img = img.unsqueeze(0)  # Add batch dimension → shape becomes [1, 3, 112, 112]
    return img

# Function to build gallery embeddings from images in a folder
def build_gallery_simple(model, gallery_path, device):

    model.eval()  # Set model to evaluation mode (important for BatchNorm/Dropout)
    gallery = {}  # Dictionary to store {person_name: embedding}

    with torch.no_grad():  # Disable gradient computation for inference

        # Loop through all image files in gallery folder
        for img_name in os.listdir(gallery_path):

            # Create full file path
            img_path = os.path.join(gallery_path, img_name)

            # Read image using OpenCV
            img = cv2.imread(img_path)

            # Skip file if image failed to load
            if img is None:
                continue

            # Resize image to match model input size (112x112)
            img = cv2.resize(img, (112, 112))

            # Preprocess image and move it to device (GPU/CPU)
            img = preprocess(img).to(device)

            # Generate embedding from the model
            emb = model(img)

            # Remove file extension (.jpg/.png) to use filename as identity label
            person_name = os.path.splitext(img_name)[0]

            # Store embedding in gallery dictionary
            gallery[person_name] = emb

    return gallery  # Return dictionary containing all identity embeddings


# Build gallery from images inside the folder
gallery = build_gallery_simple(model, "/content/gallery", device)

# Print all stored identity names
print(gallery.keys())

dict_keys(['000025', '000023', '000015'])


In [25]:
print("Images found:", os.listdir("/content/gallery"))

Images found: ['000025.jpg', '000023.jpg', '000015.jpg']


In [26]:
gallery = build_gallery_simple(model, "/content/gallery", device)

print(gallery.keys())

dict_keys(['000025', '000023', '000015'])


In [31]:
import cv2

def identify(model, query_img, gallery, device):

    model.eval()  # Set model to evaluation mode (important for BatchNorm/Dropout)

    with torch.no_grad():  # Disable gradient computation for inference

        # Resize query image to match model input size (112x112)
        query_img = cv2.resize(query_img, (112, 112))

        # Preprocess image (normalize, convert to tensor, add batch dimension)
        # Then move it to device (GPU/CPU)
        query_img = preprocess(query_img).to(device)

        # Generate embedding for the query image
        query_emb = model(query_img)

        # Initialize minimum distance as infinity
        min_dist = float('inf')

        # Variable to store predicted identity
        predicted_person = None

        # Compare query embedding with each gallery embedding
        for person_name, gallery_emb in gallery.items():

            # Compute Euclidean (L2) distance between query and gallery embedding
            dist = torch.norm(query_emb - gallery_emb, dim=1).item()

            # Update minimum distance and predicted identity if smaller distance found
            if dist < min_dist:
                min_dist = dist
                predicted_person = person_name

    # Return predicted identity and corresponding minimum distance
    return predicted_person, min_dist

In [38]:
query = cv2.imread("/content/gallery/000025.jpg")

pred_name, distance = identify(model, query, gallery, device)

print("Predicted:", pred_name)
print("Distance:", distance)

Predicted: 000025
Distance: 0.0


In [ ]:
import matplotlib.pyplot as plt
import os

# Load the original query image (it's already loaded in 'query')
query_display_img = cv2.cvtColor(query, cv2.COLOR_BGR2RGB)

# Load the identified image from the gallery
identified_img_path = os.path.join("/content/gallery", pred_name + ".jpg")
identified_display_img = cv2.imread(identified_img_path)
identified_display_img = cv2.cvtColor(identified_display_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(query_display_img)
plt.title("Query Image")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(identified_display_img)
plt.title(f"Identified as: {pred_name} (Distance: {distance:.2f})")
plt.axis('off')

plt.show()